<a href="https://colab.research.google.com/github/HYEONis/LLM-AI-Deep-Learning/blob/main/LLM_pratice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Chapter 04

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ExampleDeepNeuralNetwork(nn.Module):
  def __init__(self, layer_sizes, use_shortcut):
    super().__init__()
    self.use_shortcut = use_shortcut
    self.layers = nn.ModuleList([
        nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), nn.GELU()),
        nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), nn.GELU()),
        nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), nn.GELU()),
        nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), nn.GELU()),
        nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), nn.GELU()) #5개의 층 생성
    ])

  def forward(self, x):
    for layer in self.layers:
        layer_output = layer(x) #현재 층의 출력을 계산
        if self.use_shortcut and x.shape == layer_output.shape: #숏컷 연결을 적용할 수 있는지 확인함
          x = x + layer_output
        else:
          x = layer_output
    return x

layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1.,0.,-1.]])
torch.manual_seed(123) #초기 가중치를 재현할 수 있도록 랜덤 시드를 배치함
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)

def print_gradients(model, x):
  output = model(x) #정방향 계산
  target = torch.tensor([[0.]])

  loss = nn.MSELoss()
  loss = loss(output, target) #타깃과 출력에 가까운 정도를 기반으로 손실을 계산한다.

  loss.backward() #그레이디언트 계산을 위한 역전파
  for name, param in model.named_parameters():
    if 'weight'in name:
      print(f"{name}의 평균 그레이디언트는 {param.grad.abs().mean().item()}이다.")
      print_gradients(model_without_shortcut, sample_input)

      torch.manual_seed(123)
      model_with_shortcut = ExampleDeepNetwork(layer_sizes, use_shortcut = True)
      print_gradients(model_with_shortcut, sample_input)

In [ ]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

In [ ]:
from previous_chapters import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

ModuleNotFoundError: No module named 'previous_chapters'

In [7]:
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

    self.final_norm = LayerNorm(cfg["emb_dim"])

    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias = False)

  def forward(self, x):
      batch_size, seq_len = x.shape
      tok_emb = self.tok_emb(x)

      pos_embeds = self.pos_emb(torch.arange(seq_len, device = x.device)) #장치 설정을 통해서 입력 데이터가 어디있는지 따라 CPU나 GPU에 훈련


      x = tok_emb + pos_embeds
      x = self.drop_emb(x)
      x = self.trf_blocks(x)
      x = self.final_norm(x)
      logits = self.out_head(x)

      return logits

      torch.manual_seed(123)
      model = GPTModel(GPT_CONFIG_124M)
      out = model(batch)

      print("입력배치 \n", batch)
      print("\n 출력크기", out.shape)
      print(out)

In [17]:
import torch
import torch.nn as nn


class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))
        self.shift = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.linear = nn.Linear(cfg["emb_dim"], cfg["emb_dim"])
        self.norm = LayerNorm(cfg["emb_dim"])

    def forward(self, x):
        return self.norm(self.linear(x))

# GPT Model

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x):
        batch_size, seq_len = x.shape

        tok_emb = self.tok_emb(x)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=x.device))

        x = tok_emb + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        logits = self.out_head(x)
        return logits


GPT_CONFIG_124M = {
    "vocab_size": 50257, # Updated vocab_size to match gpt2 tokenizer
    "context_length": 32,
    "emb_dim": 64,
    "n_layers": 2,
    "drop_rate": 0.1
}

torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)

batch = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 10))

out = model(batch)

total_params= sum(p.numel() for p in model.parameters())
print(f"총 파라미터의 갯수:{total_params:,}")

print("입력배치:\n", batch)
print("\n출력 크기:", out.shape)
print(out)

print(model.tok_emb.weight.shape)
print(model.out_head.weight.shape)

총 파라미터의 갯수:6,443,648
입력배치:
 tensor([[  728, 25863, 27194, 32442, 17577, 15106, 19752,  9328, 23003, 48718],
        [22518, 40164, 33151, 12990,  7665, 35292,   201, 16274,  1072, 34149]])

출력 크기: torch.Size([2, 10, 50257])
tensor([[[-0.1228,  0.0622,  0.7591,  ...,  0.7535,  0.6199, -0.2358],
         [ 1.2598, -0.3186,  0.5942,  ..., -0.4556, -0.6482,  0.2028],
         [-0.3780, -0.0324, -0.4065,  ...,  0.6593, -1.4442,  0.1107],
         ...,
         [-0.8424,  0.4670, -0.0170,  ...,  0.1057,  0.3381, -0.2152],
         [ 0.4998,  0.4736, -0.1502,  ..., -0.7727,  0.9092,  0.8966],
         [-0.0296,  0.9940, -0.0371,  ..., -0.5569,  0.0789, -0.0307]],

        [[-0.0056,  0.2658,  0.4310,  ...,  0.3420,  1.2384,  0.6886],
         [ 1.2422, -0.2757, -0.3836,  ..., -0.6120,  0.5057, -0.1019],
         [-0.1660, -0.4227, -0.6293,  ...,  0.7277, -1.5739, -0.3671],
         ...,
         [-1.2011,  0.4191, -0.1142,  ...,  0.0165,  0.5257, -0.5304],
         [-0.4873, -0.2483, -0.0860,

In [18]:
import tiktoken
import torch

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

tokenizer = tiktoken.get_encoding("gpt2") # Initialize the tokenizer here
start_context = "Hello, i am"
encoded = tokenizer.encode(start_context)
print("인코딩된 id :", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

model.eval()
out = generate_text_simple(model = model,
                           idx = encoded_tensor,
                           max_new_tokens=6,
                           context_size=GPT_CONFIG_124M["context_length"]
)

print("출력", out)
print("출력 길이 :", len(out[0]))

인코딩된 id : [15496, 11, 1312, 716]
encoded_tensor.shape: torch.Size([1, 4])
출력 tensor([[15496,    11,  1312,   716, 13423,  8622, 12566, 41835, 21809,  2903]])
출력 길이 : 10
